# 预定义状态 MessagesState

In [1]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState

# 导入环境变量，覆盖写入
load_dotenv(override=True)

# 0.连接大模型
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 1、定义状态
class OverAllState(MessagesState):
    # 添加一些自定义状态
    username: str
    output: str


# 2、定义节点
def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好，我是" + state["username"])]
    }


# 大语言模型节点
# 为什么讲预定义状态，因为其可以给大预言模型发送消息。将对话信息用预定义的messages保存起来
def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }


# 3、构建图
builder = StateGraph(state_schema=OverAllState)
# 添加节点
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
# 边
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()
result = graph.invoke({"username": "张三"})
print(result)

{'messages': [HumanMessage(content='你好，我是张三', additional_kwargs={}, response_metadata={}, id='7f8a951a-0d08-4b44-8cc0-3b7fdc96f6ad'), AIMessage(content='你好，张三！很高兴认识你。有什么我可以帮你的吗？无论是学习、生活还是工作上的问题，都可以随时和我聊聊哦！😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 8, 'total_tokens': 39, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '986da30c-a645-49bf-92a2-1e6225dfa3f7', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f7314-8b8f-7d61-836f-a37557b30923-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 31, 'total_tokens': 39, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'username': '张三', 'outpu